# 03 — Тестирование /predict endpoint

Ноутбук демонстрирует работу endpoint `/api/predict` двумя способами:
1. **Прямой вызов модуля** (работает всегда, без запущенного сервиса)
2. **HTTP-запрос к запущенному сервису** (требует `uvicorn src.main:app`)

> Для HTTP-теста из Jupyter используется `nest_asyncio` — он позволяет вызывать `asyncio` внутри уже запущенного event loop.

In [1]:
import json
import sys
import asyncio
from pathlib import Path

sys.path.insert(0, "..")

# nest_asyncio позволяет asyncio.run() / await внутри Jupyter
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    # Установить: pip install nest_asyncio
    print("nest_asyncio не установлен, HTTP-тест будет пропущен")

## 1. Прямое тестирование модуля (без HTTP, работает всегда)

In [2]:
import os
os.environ.setdefault("TASK_PROVIDER", "rules")  # без API-ключа — rule-based
os.environ.setdefault("DATABASE_URL", "sqlite:///./test_nb.db")

from src.services.task_extraction import extract_tasks_rule_based, extract_tasks
from src.services.evaluation import evaluate_tasks

# Загружаем тестовый транскрипт из gold-данных
GOLD_DIR = Path("../data/gold")
f = GOLD_DIR / "ES2002a.json"
d = json.loads(f.read_text())
transcript = d.get("transcript", "")[:3000]
gold_tasks = d.get("tasks", [])

print(f"Транскрипт: {len(transcript)} символов")
print(f"Gold задачи ({len(gold_tasks)}):")
for t in gold_tasks:
    print(f"  - {t['description'][:80]}")

Транскрипт: 3000 символов
Gold задачи (3):
  - Work on the actual working design of the remote control.
  - Define technical functions and determine what the user interface will actually d
  - Analyze product requirements and determine what they need to fulfill.


In [3]:
# Rule-based экстрактор
pred_tasks = extract_tasks_rule_based(transcript)
m = evaluate_tasks(pred_tasks, gold_tasks)

print(f"Rule-based: найдено {len(pred_tasks)} задач")
print(f"  Task F1:   {m['task_set_f1']:.3f}")
print(f"  Precision: {m['task_set_precision']:.3f}")
print(f"  Recall:    {m['task_set_recall']:.3f}")
print(f"  Matched:   {m['matched_tasks']}/{m['gold_tasks']}")
print()
print("Найденные задачи:")
for t in pred_tasks[:5]:
    print(f"  - {t['description'][:80]}")

[TASK-CLS] Loaded artifact is not usable, retraining


Rule-based: найдено 12 задач
  Task F1:   0.133
  Precision: 0.083
  Recall:    0.333
  Matched:   1/3

Найденные задачи:
  - Hi, I'm David and I'm supposed to be an industrial designer
  - Um so we're designing a new remote control and um Oh I have to record who's here
  - Um yeah so des uh design a new remote control.
  - Um and so there are three different stages to the design.
  - , right, well basically um high priority for any animal for me is that they be w


In [4]:
# Async-вызов extract_tasks (основная функция, используемая в /predict)
async def run_extract():
    result = await extract_tasks(
        transcript,
        return_debug=True,
        meeting_ref="ES2002a",
        language="en",
        duration_sec=d.get("duration_sec", 0),
    )
    return result

tasks, debug = asyncio.run(run_extract())
print(f"extract_tasks: {len(tasks)} задач")
print(f"  provider:      {debug.get('provider')}")
print(f"  fallback_used: {debug.get('fallback_used')}")
print(f"  parse_stage:   {debug.get('parse_stage')}")

extract_tasks: 12 задач
  provider:      rules
  fallback_used: True
  parse_stage:   None


## 2. HTTP-тест endpoint /predict (нужен запущенный сервис)

Запустите сервис отдельно:
```bash
cd project
uvicorn src.main:app --reload --host 0.0.0.0 --port 8000
```

**WSL-пользователи:** сервис запускается на Windows, из WSL нужно использовать IP хоста Windows,
а не `localhost`. Узнайте IP: `cat /etc/resolv.conf | grep nameserver` или `ip route | grep default`.

In [5]:
import httpx

# Измените если сервис запущен на другом хосте/порту
# Для WSL: замените localhost на IP Windows-хоста из /etc/resolv.conf
import os
SERVICE_URL = os.environ.get("SERVICE_URL", "http://localhost:8000")
print(f"Service URL: {SERVICE_URL}")

async def call_predict(transcript: str, meeting_ref: str = "test") -> dict:
    async with httpx.AsyncClient(timeout=30.0) as client:
        resp = await client.post(
            f"{SERVICE_URL}/api/predict",
            json={"transcript": transcript, "meeting_ref": meeting_ref, "language": "en"},
        )
        resp.raise_for_status()
        return resp.json()

# Короткий тестовый транскрипт
test_text = (
    "Alice will prepare the design specification by Friday. "
    "Bob needs to review the product requirements and send feedback."
)

try:
    result = asyncio.run(call_predict(test_text, "test-01"))
    print(f"/predict ответил успешно")
    print(f"  task_count:    {result['task_count']}")
    print(f"  provider:      {result.get('provider')}")
    print(f"  fallback_used: {result.get('fallback_used')}")
    print()
    for i, t in enumerate(result.get("tasks", []), 1):
        print(f"  [{i}] {t.get('description','')[:80]}")
except Exception as e:
    print(f"Сервис недоступен: {e}")
    print("   Убедитесь что uvicorn запущен и порт 8000 открыт.")
    print("   WSL: используйте IP Windows-хоста вместо localhost")

Service URL: http://localhost:8000
/predict ответил успешно
  task_count:    2
  provider:      rules
  fallback_used: True

  [1] Alice will prepare the design specification by Friday.
  [2] Bob needs to review the product requirements and send feedback.


In [6]:
# Тест с полным AMI-транскриптом + сравнение с gold
try:
    result = asyncio.run(call_predict(transcript, "ES2002a"))
    pred_from_api = result.get("tasks", [])
    m_api = evaluate_tasks(pred_from_api, gold_tasks)
    
    print(f"/predict на ES2002a:")
    print(f"  Задач найдено: {result['task_count']}")
    print(f"  Task F1:       {m_api['task_set_f1']:.3f}")
    print(f"  Precision:     {m_api['task_set_precision']:.3f}")
    print(f"  Recall:        {m_api['task_set_recall']:.3f}")
    print(f"  Matched:       {m_api['matched_tasks']}/{m_api['gold_tasks']}")
except Exception as e:
    print(f"Сервис недоступен: {e}")

/predict на ES2002a:
  Задач найдено: 12
  Task F1:       0.133
  Precision:     0.083
  Recall:        0.333
  Matched:       1/3


## 3. Тест /health endpoint

In [7]:
async def check_health():
    async with httpx.AsyncClient(timeout=5.0) as client:
        resp = await client.get(f"{SERVICE_URL}/api/health")
        return resp.json()

try:
    health = asyncio.run(check_health())
    print(f"Health: {health}")
except Exception as e:
    print(f"Health check failed: {e}")

Health: {'status': 'ok', 'timestamp': 1779204600.119421, 'db': 'ok', 'service': 'meeting-secretary', 'version': '1.0.0'}


## 4. Итог

Endpoint `/api/predict`:
- принимает `transcript`, `language`, `duration_sec`, `meeting_ref`
- возвращает список задач + метаданные (`provider`, `model`, `fallback_used`, `parse_stage`)
- при отсутствии `OPENROUTER_API_KEY` автоматически использует rule-based fallback
- тестируется автоматически в `tests/test_predict_endpoint.py` (7 тестов)

## 5. Матрица покрытия тестов

| Способ | Что тестируется | Нужен сервис? |
|---|---|---:|
| Прямой вызов `extract_tasks_rule_based` | Rule-based pipeline | ❌ |
| `asyncio.run(extract_tasks(...))` | Основная async-функция + LLM fallback | ❌ |
| `POST /api/predict` (HTTP) | FastAPI endpoint, маршрутизация | ✅ |
| `POST /api/predict` с gold (HTTP) | End-to-end F1 через API | ✅ |
| `GET /api/health` | DB + сервис статус | ✅ |
| `pytest tests/` | Автоматические тесты (7 штук) | ❌/✅ |

**Запуск автотестов:**
```bash
cd project
pytest tests/ -v
```

**Переменная окружения** для WSL / удалённого хоста:
```bash
export SERVICE_URL=http://172.x.x.x:8000  # IP из /etc/resolv.conf
jupyter nbconvert --to notebook --execute 03_predict_endpoint_test.ipynb
```
